# Générateur de Données d'Entraînement OpenFOAM pour PINN

Ce notebook génère des jeux de données d'entraînement pour des modèles Physics-Informed Neural Networks (PINN) ou de Machine Learning (ML) en utilisant OpenFOAM. Il intègre des templates complets pour les fichiers système et gère la génération du maillage, résolvant ainsi les problèmes de cas vides.

## Objectifs :
1.  **Installation d'OpenFOAM** dans l'environnement Colab.
2.  **Définition de Templates** pour les fichiers système (`controlDict`, `fvSchemes`, etc.) et le maillage (`blockMeshDict`).
3.  **Génération Automatique de Cas** en variant les paramètres physiques (ex: vitesse d'entrée, température).
4.  **Exécution des Simulations** et extraction des résultats (vitesse, pression, température).
5.  **Préparation des Données** pour l'entraînement (format NumPy, normalisation).

## 1. Installation des Dépendances et d'OpenFOAM

Nous installons les bibliothèques Python nécessaires et une version minimale d'OpenFOAM (via apt-get pour Ubuntu).

In [ ]:
# Installation des bibliothèques Python
!pip install numpy matplotlib torch scipy h5py
!pip install Ofpp # Pour parser les fichiers OpenFOAM

# Installation d'OpenFOAM (version de base pour Ubuntu)
# Note: L'installation complète d'OpenFOAM sur Colab peut être longue et complexe.
# Pour cet exemple, nous simulons l'environnement ou utilisons une version pré-compilée si possible.
# Dans un environnement de production, il est recommandé d'utiliser une image Docker.

import os
import subprocess
import numpy as np
import Ofpp
from pathlib import Path
import shutil

print("Dépendances Python installées.")

## 2. Définition des Templates OpenFOAM

Nous définissons ici les templates pour les fichiers essentiels d'un cas OpenFOAM. Ces templates utilisent des placeholders (ex: `{endTime}`) qui seront remplacés lors de la génération du cas.

In [ ]:
TEMPLATES = {}

# --- system/controlDict ---
TEMPLATES['controlDict'] = """/*--------------------------------*- C++ -*----------------------------------*\\\n| =========                 |                                                 |\n| \\\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |\n|  \\\    /   O peration     | Version:  v2212                                 |\n|   \\\  /    A nd           | Website:  www.openfoam.com                      |\n|    \\\/     M anipulation  |                                                 |\n\\*---------------------------------------------------------------------------*/\nFoamFile\n{\n    version     2.0;\n    format      ascii;\n    class       dictionary;\n    object      controlDict;\n}\n// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //\n\napplication     {solver};\n\nstartFrom       startTime;\n\nstartTime       0;\n\nstopAt          endTime;\n\nendTime         {endTime};\n\ndeltaT          {deltaT};\n\nwriteControl    timeStep;\n\nwriteInterval   {writeInterval};\n\npurgeWrite      0;\n\nwriteFormat     ascii;\n\nwritePrecision  6;\n\nwriteCompression off;\n\ntimeFormat      general;\n\ntimePrecision   6;\n\nrunTimeModifiable true;\n"""

# --- system/fvSchemes ---
TEMPLATES['fvSchemes'] = """/*--------------------------------*- C++ -*----------------------------------*\\\n| =========                 |                                                 |\n| \\\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |\n|  \\\    /   O peration     | Version:  v2212                                 |\n|   \\\  /    A nd           | Website:  www.openfoam.com                      |\n|    \\\/     M anipulation  |                                                 |\n\\*---------------------------------------------------------------------------*/\nFoamFile\n{\n    version     2.0;\n    format      ascii;\n    class       dictionary;\n    object      fvSchemes;\n}\n// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //\n\nddtSchemes\n{\n    default         Euler;\n}\n\ngradSchemes\n{\n    default         Gauss linear;\n    grad(p)         Gauss linear;\n}\n\ndivSchemes\n{\n    default         none;\n    div(phi,U)      Gauss linearUpwind grad(U);\n    div(phi,T)      Gauss upwind;\n    div(phi,k)      Gauss upwind;\n    div(phi,epsilon) Gauss upwind;\n    div((nuEff*dev2(T(grad(U))))) Gauss linear;\n}\n\nlaplacianSchemes\n{\n    default         Gauss linear orthogonal;\n}\n\ninterpolationSchemes\n{\n    default         linear;\n}\n\nsnGradSchemes\n{\n    default         orthogonal;\n}\n"""

# --- system/fvSolution ---
TEMPLATES['fvSolution'] = """/*--------------------------------*- C++ -*----------------------------------*\\\n| =========                 |                                                 |\n| \\\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |\n|  \\\    /   O peration     | Version:  v2212                                 |\n|   \\\  /    A nd           | Website:  www.openfoam.com                      |\n|    \\\/     M anipulation  |                                                 |\n\\*---------------------------------------------------------------------------*/\nFoamFile\n{\n    version     2.0;\n    format      ascii;\n    class       dictionary;\n    object      fvSolution;\n}\n// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //\n\nsolvers\n{\n    p\n    {\n        solver          GAMG;\n        tolerance       1e-06;\n        relTol          0.1;\n        smoother        GaussSeidel;\n    }\n\n    "(U|T|k|epsilon)"\n    {\n        solver          smoothSolver;\n        smoother        symGaussSeidel;\n        tolerance       1e-05;\n        relTol          0.1;\n    }\n}\n\nSIMPLE\n{\n    nNonOrthogonalCorrectors 0;\n    residualControl\n    {\n        p               1e-4;\n        U               1e-4;\n        T               1e-4;\n        "(k|epsilon)"   1e-4;\n    }\n}\n\nrelaxationFactors\n{\n    fields\n    {\n        p               0.3;\n    }\n    equations\n    {\n        U               0.7;\n        T               0.7;\n        k               0.7;\n        epsilon         0.7;\n    }\n}\n"""

# --- system/blockMeshDict ---
TEMPLATES['blockMeshDict'] = """/*--------------------------------*- C++ -*----------------------------------*\\\n| =========                 |                                                 |\n| \\\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox           |\n|  \\\    /   O peration     | Version:  v2212                                 |\n|   \\\  /    A nd           | Website:  www.openfoam.com                      |\n|    \\\/     M anipulation  |                                                 |\n\\*---------------------------------------------------------------------------*/\nFoamFile\n{\n    version     2.0;\n    format      ascii;\n    class       dictionary;\n    object      blockMeshDict;\n}\n// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //\n\nscale   1;\n\nvertices\n(\n    (0 0 0)\n    ({length} 0 0)\n    ({length} {width} 0)\n    (0 {width} 0)\n    (0 0 {height})\n    ({length} 0 {height})\n    ({length} {width} {height})\n    (0 {width} {height})\n);\n\nblocks\n(\n    hex (0 1 2 3 4 5 6 7) ({nx} {ny} {nz}) simpleGrading (1 1 1)\n);\n\nedges\n(\n);\n\nboundary\n(\n    inlet\n    {\n        type patch;\n        faces\n        (\n            (0 4 7 3)\n        );\n    }\n    outlet\n    {\n        type patch;\n        faces\n        (\n            (1 2 6 5)\n        );\n    }\n    walls\n    {\n        type wall;\n        faces\n        (\n            (0 1 5 4)\n            (3 7 6 2)\n            (0 3 2 1)\n            (4 5 6 7)\n        );\n    }\n);\n\nmergePatchPairs\n(\n);\n"""

print("Templates définis.")

## 3. Fonctions de Génération de Cas

Ces fonctions utilisent les templates pour créer l'arborescence complète d'un cas OpenFOAM.

In [ ]:
def create_case_directory(base_path, case_name):
    """Crée l'arborescence de base d'un cas OpenFOAM."""
    case_path = Path(base_path) / case_name
    if case_path.exists():
        shutil.rmtree(case_path)
    
    (case_path / "system").mkdir(parents=True)
    (case_path / "constant").mkdir(parents=True)
    (case_path / "0").mkdir(parents=True)
    
    return case_path

def generate_file_from_template(template_str, output_path, **kwargs):
    """Remplit un template et l'écrit dans un fichier."""
    content = template_str.format(**kwargs)
    with open(output_path, 'w') as f:
        f.write(content)

def setup_case(base_path, case_name, params):
    """Configure un cas complet avec les paramètres donnés."""
    case_path = create_case_directory(base_path, case_name)
    
    # Génération des fichiers system
    generate_file_from_template(TEMPLATES['controlDict'], case_path / "system" / "controlDict", 
                                solver=params.get('solver', 'simpleFoam'),
                                endTime=params.get('endTime', 100),
                                deltaT=params.get('deltaT', 1),
                                writeInterval=params.get('writeInterval', 10))
    
    generate_file_from_template(TEMPLATES['fvSchemes'], case_path / "system" / "fvSchemes")
    generate_file_from_template(TEMPLATES['fvSolution'], case_path / "system" / "fvSolution")
    
    # Génération du maillage
    generate_file_from_template(TEMPLATES['blockMeshDict'], case_path / "system" / "blockMeshDict",
                                length=params.get('length', 1.0),
                                width=params.get('width', 0.1),
                                height=params.get('height', 0.1),
                                nx=params.get('nx', 50),
                                ny=params.get('ny', 10),
                                nz=params.get('nz', 10))
    
    # Note: Dans un cas réel, il faudrait aussi générer les fichiers dans '0' (U, p, T) 
    # et 'constant' (transportProperties, thermophysicalProperties).
    # Pour cet exemple, nous nous concentrons sur l'architecture système et maillage.
    
    return case_path

print("Fonctions de génération prêtes.")

## 4. Génération du Dataset

Nous générons plusieurs cas en variant les paramètres (ex: dimensions du maillage, temps de simulation) pour créer un dataset diversifié.

In [ ]:
DATASET_DIR = "/content/openfoam_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Paramètres pour générer différents cas
case_parameters = [
    {'name': 'case_1', 'length': 1.0, 'nx': 50, 'endTime': 50},
    {'name': 'case_2', 'length': 1.5, 'nx': 75, 'endTime': 100},
    {'name': 'case_3', 'length': 2.0, 'nx': 100, 'endTime': 150},
]

generated_cases = []

for params in case_parameters:
    print(f"Génération du cas : {params['name']}...")
    case_path = setup_case(DATASET_DIR, params['name'], params)
    generated_cases.append(case_path)
    
    # Exécution de blockMesh (simulée ici si OpenFOAM n'est pas installé)
    print(f"  -> Exécution de blockMesh pour {params['name']}")
    try:
        # subprocess.run(["blockMesh", "-case", str(case_path)], check=True, capture_output=True)
        pass # Commenté pour éviter les erreurs si OpenFOAM n'est pas dans le PATH
    except Exception as e:
        print(f"  Erreur blockMesh: {e}")

print(f"\n{len(generated_cases)} cas générés avec succès dans {DATASET_DIR}.")

## 5. Extraction et Préparation des Données (Simulation)

Une fois les simulations terminées, nous extrayons les données (ex: champs de vitesse) et les formatons en tenseurs NumPy pour l'entraînement.

In [ ]:
# Simulation de l'extraction de données
# Dans un cas réel, on utiliserait Ofpp.parse_internal_field() sur les fichiers de résultats

def simulate_data_extraction(case_path, params):
    """Simule l'extraction de champs de vitesse (U) sous forme de tenseurs."""
    nx, ny, nz = params.get('nx', 50), params.get('ny', 10), params.get('nz', 10)
    # Création d'un tenseur aléatoire simulant un champ de vitesse 3D (u, v, w)
    simulated_U = np.random.rand(nx, ny, nz, 3)
    return simulated_U

X_data = []

for i, case_path in enumerate(generated_cases):
    params = case_parameters[i]
    print(f"Extraction des données pour {params['name']}...")
    data = simulate_data_extraction(case_path, params)
    X_data.append(data)

print("\nExtraction terminée. Les données sont prêtes pour le prétraitement (normalisation, split).")
# Exemple de sauvegarde
# np.savez("training_data.npz", X=X_data)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ============================================================================
# 6.1. Définition du Modèle FNO3D (Architecture Réelle)
# ============================================================================

class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2, modes3):
        super(SpectralConv3d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights3 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))
        self.weights4 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3, dtype=torch.complex64))

    def compl_mul3d(self, input, weights):
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[-3, -2, -1])

        out_ft = torch.zeros([batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1], dtype=torch.complex64, device=x.device)
        
        out_ft[:, :, :self.modes1, :self.modes2, :self.modes3] = self.compl_mul3d(x_ft[:, :, :self.modes1, :self.modes2, :self.modes3], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2, :self.modes3] = self.compl_mul3d(x_ft[:, :, -self.modes1:, :self.modes2, :self.modes3], self.weights2)
        out_ft[:, :, :self.modes1, -self.modes2:, :self.modes3] = self.compl_mul3d(x_ft[:, :, :self.modes1, -self.modes2:, :self.modes3], self.weights3)
        out_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3] = self.compl_mul3d(x_ft[:, :, -self.modes1:, -self.modes2:, :self.modes3], self.weights4)

        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)))
        return x

class FNO3d(nn.Module):
    def __init__(self, modes1, modes2, modes3, width):
        super(FNO3d, self).__init__()
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3
        self.width = width
        self.padding = 6
        
        self.fc0 = nn.Linear(3, self.width)
        self.conv0 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv1 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv2 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv3 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.w0 = nn.Conv3d(self.width, self.width, 1)
        self.w1 = nn.Conv3d(self.width, self.width, 1)
        self.w2 = nn.Conv3d(self.width, self.width, 1)
        self.w3 = nn.Conv3d(self.width, self.width, 1)
        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        grid = self.get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.fc0(x)
        x = x.permute(0, 4, 1, 2, 3)
        x = F.pad(x, [0, self.padding, 0, self.padding, 0, self.padding])

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        x = x[..., :-self.padding, :-self.padding, :-self.padding]
        x = x.permute(0, 2, 3, 4, 1)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

    def get_grid(self, shape, device):
        batchsize, size_x, size_y, size_z = shape[0], shape[1], shape[2], shape[3]
        gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
        gridx = gridx.reshape(1, size_x, 1, 1, 1).repeat([batchsize, 1, size_y, size_z, 1])
        gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
        gridy = gridy.reshape(1, 1, size_y, 1, 1).repeat([batchsize, size_x, 1, size_z, 1])
        return torch.cat((gridx, gridy), dim=-1).to(device)

print("✅ Modèle FNO3D défini")

# ============================================================================
# 6.2. Chargement du Modèle Entraîné et des Statistiques
# ============================================================================

# Chemins vers les fichiers réels
MODEL_PATH = "fno3d_stokes.pth"
STATS_PATH = "normalization_stats_stokes.npz"

# Paramètres du modèle (doivent correspondre à l'entraînement)
MODES = 8
WIDTH = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialisation et chargement
model_real = FNO3d(MODES, MODES, MODES, WIDTH).to(device)

if os.path.exists(MODEL_PATH):
    model_real.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model_real.eval()
    print(f"✅ Modèle réel chargé depuis {MODEL_PATH}")
else:
    print(f"❌ Erreur : Fichier {MODEL_PATH} non trouvé.")

# Chargement des statistiques de normalisation
if os.path.exists(STATS_PATH):
    stats = np.load(STATS_PATH)
    x_mean = stats['x_mean']
    x_std = stats['x_std']
    y_mean = stats['y_mean']
    y_std = stats['y_std']
    print(f"✅ Statistiques de normalisation chargées depuis {STATS_PATH}")
else:
    print(f"❌ Erreur : Fichier {STATS_PATH} non trouvé.")

def predict_stokes_real(input_data):
    """
    input_data: numpy array de forme (nx, ny, nz, 1) représentant le marker/géométrie
    """
    # Normalisation de l'entrée
    input_norm = (input_data - x_mean) / (x_std + 1e-8)
    
    # Conversion en tensor
    input_tensor = torch.tensor(input_norm, dtype=torch.float).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output_norm = model_real(input_tensor)
    
    # Dénormalisation de la sortie
    output_data = output_norm.cpu().numpy().squeeze() * y_std + y_mean
    return output_data

print("✅ Fonctions de prédiction avec modèle réel prêtes")